In [2]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path().resolve()
sys.path.insert(0, str(project_root))

# general dependencies
import pandas as pd
import re
import yaml

# pydantic dependencies
from pydantic import BaseModel, field_validator, PrivateAttr, model_validator, ValidationError, Field, StringConstraints
from typing import Literal
from typing import Annotated

# general functions
from utilities.general_functions import TokenMapParser, parse_valve_callout

# submodels
from submodels.station_components.base_mounted_valves import Base_Mounted_Valves_Model
from submodels.mounting_and_nameplate import Mounting_And_Nameplate_Model

# Loading data
from utilities.config import YAML_DATA


In [6]:
def get_porting_type(element,data=YAML_DATA):
    data_dict=data["sup_exh_porting_dir_and_cover_assy_symbols"][element]
    
    porting_type = data_dict['porting_type']
    pe_port_entry = data_dict['pe_port_entry']

    return data_dict, pe_port_entry, porting_type

get_porting_type('A', data=YAML_DATA)

({'field': 'sup_exh_porting_dir_and_cover_assy',
  'porting_type': '10',
  'pe_port_entry': 'U',
  'silencer': False},
 'U',
 '10')

## Submodels

### Various Valve (Station Component) Models

#### Valve Model Identification and Creation Function

In [ ]:
VALVE_MODEL_REGISTRY = {
    "base_mounted_valve": Base_Mounted_Valves_Model,
    # "x_options": Special_Valve_Model,
    # "blanking_plate": Pilot_Valve_Model,

}

def build_valve_model(symbol: str, qty: int, yaml_entry: dict, parent: SY1_EX600_MODEL):
    valve_type = yaml_entry["type"]
    model_cls = VALVE_MODEL_REGISTRY[valve_type]

    return model_cls(
        qty=qty,
        symbol=symbol,
        lt_surge_volt_sup_and_coil_type=parent.lt_surge_volt_sup_and_coil_type,
        manual_override=parent.manual_override,
        ab_port_size=parent.ab_port_size,
        **yaml_entry  # fills actuation, seal_type, etc.
    )

---

##### Mounting and Nameplate

In [6]:
class Mounting_And_Nameplate_Model(BaseModel):
    symbol: str
    catalog_value: str | None = None
    mounting: str | None = None
    sta_length: int | None = None # due to A,B,D and A0, B0, D0 - not all values are ints intially
    parent_series: str | None = None 

    @model_validator(mode="after")
    def load_yaml_and_validate(self):
        if self.symbol not in field_values['mounting_and_nameplate_symbols']:
            raise ValueError(f"Unknown mounting/nameplate symbol: {self.symbol}")

        entry = field_values["mounting_and_nameplate_symbols"][self.symbol]

        self.catalog_value = entry.get("catalog_value")
        self.mounting = entry.get("mounting")
        self.sta_length = entry.get("sta_length")

        # Example rule: series 7 cannot use mounting type "direct"
        # if self.parent_series == "7" and self.mounting == "direct":
        #     raise ValueError("Series 7 cannot use direct mounting")

        return self

## Part Validation Model

In [ ]:
SY1_EX600_TOKEN_MAP = [
    {"name": "prefix", "pattern": r"SY", "length": 2},
    {"name": "series", "pattern": r"[357]", "length": 1},
    {"name": "EX600", "pattern": r"[6]", "length": 1},
    {"name": "separator", "pattern": r"-", "length": 1},
    
    {"name": "si_unit", "pattern": r"(0|Q|N|V|E|D|F|G|W)", "length": 1},
    {"name": "endplate_type", "pattern": r"(2|3|4|5|6|7|8|9)?", "length": 1},
    {"name": "io_unit_1", "pattern": r"[A-Z1]?", "length": None},
    {"name": "io_unit_2", "pattern": r"[A-Z1]?", "length": None},
    {"name": "io_unit_3", "pattern": r"[A-Z1]?", "length": None},
    {"name": "io_unit_4", "pattern": r"[A-Z1]?", "length": None},
    {"name": "separator", "pattern": r"-", "length": 1},
    
    {"name": "lt_surge_volt_sup_and_coil_type", "pattern": r"(R|U|S|Z|T|V|M)", "length": 1},
    {"name": "manual_override", "pattern": r"(D|E|F)?", "length": None},
    {"name": "separator", "pattern": r"-", "length": 1},
    
    {"name": "valve_callout", "pattern": r'(?:(?:[2-9]|1[0-9]|2[0-4])?(?:0[DS]|[A-W][A-W]|X|Y|Z))+', "length": None},
    {"name": "separator", "pattern": r"-", "length": 1},
    
    {"name": "sup_exh_porting_dir_and_cover_assy", "pattern": r'[A-Z]', 'length': 1},
    {"name": "ab_port_size", 'pattern': r'(1[1-7]|2[1-5]|3[1-5]|4[1-5]|5[1-4]|6[1-4])', 'length': 2},
    {"name": "mounting_and_nameplate", "pattern": r'[ABD](?:0|[A-X])?', 'length': None},
    
]

# --------------------------------------------------

class SY1_EX600_MODEL(BaseModel):
    # ----- How to Order Information -----
    # -----------------------------------------------------
    prefix: Literal['SY']
    series: Literal['3', '5', '7']
    EX600: Literal['6']
    # -
    si_unit: Literal['0', 'Q', 'N', 'V', 'E', 'D', 'F', 'G', 'W']
    endplate_type: Literal['', '2', '3', '4', '5', '6', '7', '8', '9']
    io_unit_1: Annotated[str, StringConstraints(min_length=1, max_length=1, pattern=r'[A-Z1]')]
    io_unit_2: Annotated[str, StringConstraints(min_length=1, max_length=1, pattern=r'[A-Z1]')]
    io_unit_3: Annotated[str, StringConstraints(min_length=1, max_length=1, pattern=r'[A-Z1]')]
    io_unit_4: Annotated[str, StringConstraints(min_length=1, max_length=1, pattern=r'[A-Z1]')]
    # -
    lt_surge_volt_sup_and_coil_type: Literal['R', 'U', 'S', 'Z', 'T', 'V']
    manual_override: Literal['', 'D', 'E', 'F']
    # -
    valve_callout: Annotated[str, StringConstraints(min_length=2, max_length=19, pattern=r'(?:(?:[2-9]|1[0-9]|2[0-4])?(?:0[DS]|[A-W][A-W]|X|Y|Z))+')]
    # -
    sup_exh_porting_dir_and_cover_assy: Annotated[str, StringConstraints(min_length=1, max_length=1, pattern=r'[A-Z]')]
    ab_port_size: Annotated[str, StringConstraints(min_length=2, max_length=2, pattern=r'(1[1-7]|2[1-5]|3[1-5]|4[1-5]|5[1-4]|6[1-4])')]
    mounting_and_nameplate: Annotated[str, StringConstraints(min_length=0, max_length=2, pattern=r'(?:[ABD](?:0|[A-X]))?')]

    # -----------------------------------------------------
    # --- Secondary - not a part of How-To-Order fields --- 
    _parsed_valves: list = PrivateAttr(default_factory=list)
    
    # -----------------------------------------------------
    # --- Submodels- not a part of How-To-Order fields --- 
    mounting: Mounting_And_Nameplate_Model | None = None
    

    # -----------------------------------------------------
    @field_validator("valve_callout")
    def validate_and_parse_valve_callout(cls, v, info):
        try:
            parsed = parse_valve_callout(v, valid_symbols=set(YAML_DATA["valve_symbols"].keys()))
        except ValueError as e:
            raise ValueError(f"Invalid valve callout: {e}")

        # Attach parsed valves to the model instance
        info.data["_parsed_valves"] = parsed
        return v

    # returning the dictionary of quantities and valves/various components configured in the valve callout section
    @property
    def valve_list(self):
        return self._parsed_valves
    
    # calculating the total number of valves configured on the manifold
    @property
    def num_of_stations(self):
        return sum(int(ele["qty"]) for ele in self._parsed_valves)
    
    def build_part_number(self) -> str:
        return (
            f"{self.prefix}{self.series}{self.EX600}"
            f"-{self.si_unit}{self.endplate_type}{self.io_unit_1}{self.io_unit_2}{self.io_unit_3}{self.io_unit_4}"
            f"-{self.lt_surge_volt_sup_and_coil_type}{self.manual_override}"
            f"-{self.valve_callout}"
            f"-{self.sup_exh_porting_dir_and_cover_assy}{self.ab_port_size}{self.mounting_and_nameplate}"
        )
    

    # ----- SUB MODELS -----
    @model_validator(mode="after")
    def build_submodels(self):
            self.mounting = Mounting_And_Nameplate_Model(
                symbol=self.mounting_and_nameplate,
                parent_series=self.series
            )
            # self.valves = Base_Mounted_Valves_Model(
            #     lt_surge_volt_sup_and_coil_type=self.lt_surge_volt_sup_and_coil_type,
            #     manual_override = self.manual_override,
            #     ab_port_size=self.ab_port_size
            # )


            return self
    

## Running Model

In [20]:
from IPython.display import display

part_number = "SY36-Q2AAAA-RD-2AB2AT3AAX2AE5BB2AA-A11D"

model = SY1_EX600_TOKEN_MAP
token_map = SY1_EX600_TOKEN_MAP

tokens = {}
try:
    print("Parsing:", part_number)
    parser = TokenMapParser(token_map)
    tokens = parser.parse(part_number)
    #valve = model(**tokens) #type: ignore
    manifold = SY1_EX600_MODEL(**tokens)

    validator_df = pd.DataFrame(manifold.model_dump().items(), columns=["Field", "Value"])
    print("✅ Part number is valid.\n")
    # print(validator_df)
    display(validator_df)
    print('-------------')
    # print(manifold.valves)
    display(pd.DataFrame(manifold.valve_list))
    

# PyDantic Model is Throwing Error
except ValidationError as e:
    print("Validation error:")
    print("---------------------")
    for err in e.errors():
        message = err["msg"].removeprefix("Value error, ")
        print(f"{message}")

    if "tokens" in locals():
        print("\nParsed Tokens")
        print(pd.DataFrame([tokens]))

# Parser is Throwing Error
except ValueError as e:
    print(f"Parse error: {e}")
    if "tokens" in locals():
        print("Partial Tokens Extracted")
        print(pd.DataFrame([tokens]).T)
        
    else:
        print("Parsing failed before any tokens could be generated.")

Parsing: SY36-Q2AAAA-RD-2AB2AT3AAX2AE5BB2AA-A11D
✅ Part number is valid.



c:\Users\marty\AppData\Local\Programs\Python\Python313\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value='standard', input_type=str])
  return self.__pydantic_serializer__.to_python(


,Field,Value
0,prefix,SY
1,series,3
2,EX600,6
3,si_unit,Q
4,endplate_type,2
5,io_unit_1,A
6,io_unit_2,A
7,io_unit_3,A
8,io_unit_4,A
9,lt_surge_volt_sup_and_coil_type,R


-------------


,qty,symbol
0,2,AB
1,2,AT
2,3,AA
3,1,X
4,2,AE
5,5,BB
6,2,AA


In [18]:
manifold.mounting.sta_length

'standard'

---

---

### Loading YAML symbol lists into dataframe

In [7]:
import yaml
with open("field_values.yaml", "r") as f:
    data = yaml.safe_load(f)

symbols = data["valve_symbols"]

df = pd.DataFrame.from_dict(symbols, orient="index").reset_index()
df = df.rename(columns={"index": "symbol"})
df

,symbol,field,actuation,type
0,0D,valve_callout,"NO VALVES, DOUBLE WIRE",N/A
1,0S,valve_callout,"NO VALVES, SINGLE WIRE",N/A
2,AA,valve_callout,2 POS SGL,RUBBER SEAL
3,AB,valve_callout,2 POS DBL,RUBBER SEAL
4,AC,valve_callout,3 POS CC,RUBBER SEAL
...,...,...,...,...
321,PG,valve_callout,2 POS DBL,X25
322,PH,valve_callout,2 POS DBL,"X25, -1 FITTING SIZE"
323,PI,valve_callout,2 POS DBL,"X25, -2 FITTING SIZE"
324,PJ,valve_callout,2 POS DBL,"X25, -3 FITTING SIZE"


In [1]:
vals = [{'qty': 2, 'symbol': 'AB'}, {'qty': 2, 'symbol': 'AT'}, {'qty': 3, 'symbol': 'AA'}, {'qty': 1, 'symbol': 'X'}, 
        {'qty': 2, 'symbol': 'AE'}, {'qty': 5, 'symbol': 'BB'}, {'qty': 2, 'symbol': 'AA'}]

valve_symbols = [ele["symbol"] for ele in vals]
valve_symbols

['AB', 'AT', 'AA', 'X', 'AE', 'BB', 'AA']